# Natural language processing and large language models

*Data Science and AI, Module 8. Slides: `Module_8.pdf`.*

Keep the slides open on the right. Every heading below carries the slide
numbers it goes with, so the two stay in step.

Natural language processing (NLP) is about getting a computer to do useful
work with text people wrote for other people. By the end of this notebook
you will have taken text apart with patterns, pulled it off a web page,
cleaned it, scored its mood, trained a model to sort it, and looked inside
the dice roll that makes a large language model (LLM) write.

**How each section works.** A question comes first. Make a guess out loud or
in the chat before the cell runs. Then run it and compare.

**Four kinds of text.**

- Small **made-up** examples, always labelled as made up.
- The **Titanic passenger list** from the ML challenge (`titanic_train.csv`),
  because the names hide something useful.
- **2,000 film reviews**, 1,000 positive and 1,000 negative, from Bo Pang and
  Lillian Lee, "A Sentimental Education", Proceedings of the ACL, 2004
  (polarity dataset v2.0, distributed with NLTK with the authors'
  permission).
- One live web page, `books.toscrape.com`, a site built for scraping
  practice. Its prices are random and mean nothing.

The notebook is in six parts, one per section of the deck. Each part loads
what it needs, so you can start at any part.

## Before the first session: one-off downloads

The `dsai` environment already has NLTK and spaCy. Both need some data
files the first time. Run the cell below once; afterwards it takes a second.

spaCy's English model is installed from a terminal, not from Python. If the
cell says it is missing, open the Anaconda Prompt (Windows) or Terminal
(Mac), run `conda activate dsai`, then the command the cell prints.

In [1]:
import nltk
import spacy

# Each download is skipped quickly if it is already on this computer.
nltk.download("vader_lexicon", quiet=True)   # word list for sentiment (Part 4)
nltk.download("stopwords", quiet=True)       # common words (Part 3)
nltk.download("wordnet", quiet=True)         # dictionary for lemmas (Part 3)
nltk.download("movie_reviews", quiet=True)   # the 2,000 film reviews

print("nltk", nltk.__version__)
print("spacy", spacy.__version__)

if spacy.util.is_package("en_core_web_sm"):
    print("spaCy English model: installed")
else:
    print("spaCy English model missing. Run this once in a terminal:")
    print("    python -m spacy download en_core_web_sm")    

nltk 3.10.2
spacy 3.8.14
spaCy English model: installed


# Part 1: slides 1-18

What NLP is, how text is stored, and regular expressions.

In [2]:
import re
import unicodedata
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score

DATA_FOLDER = "data/"

## Slides 3-6: what NLP does

Two broad jobs:

- **understanding** text (NLU): what is this email about, who is named in
  it, is the customer angry;
- **generating** text (NLG): write the reply, the summary, the translation.

The field moved from hand-written rules, to statistics learned from large
collections of text (a **corpus**), to neural networks, to today's large
pretrained models. This notebook walks the same road: rules first (regular
expressions, a sentiment word list), then learned models, then LLMs.

## NLP key tasks, on one sentence

Take one short piece of text and see what each task on the slide does to it:

> Jacinda Ardern studied at the University of Waikato. The course was great!

**Text processing: taking the text apart**

| Task | What it does | On our text |
|---|---|---|
| Sentences | split the text into sentences | `Jacinda Ardern studied at the University of Waikato.` and `The course was great!` |
| Words / tokens | split each sentence into pieces; punctuation counts as a piece | `Jacinda` `Ardern` `studied` `at` `the` `University` `of` `Waikato` `.` |
| Stemming | chop the ending off a word by rule; the result need not be a real word | studied → `studi`, was → `wa` |
| Lemmatisation | look the word up and return its dictionary form | studied → `study`, was → `be` |
| Part-of-speech tagging | label the job each word does in the sentence | Ardern → proper noun, studied → verb, great → adjective |

Stemming is quick but crude (`studi` and `wa` are not words). Lemmatisation is slower but gives a real word, because it knows English.

**Semantics: what the text means**

| Task | What it does | On our text |
|---|---|---|
| Named entity recognition (NER) | find names and say what kind of thing each one is | Jacinda Ardern → person, University of Waikato → organisation |
| Sentiment analysis | decide if the writer feels positive, negative or neutral | `The course was great!` → positive |
| Question answering | answer a question using the text | "Where did Jacinda Ardern study?" → the University of Waikato (needs a trained model, Part 6) |

The code cell below runs a basic pipeline over this same sentence and prints each step. Each task gets a fuller treatment later: tokens in slides 25-26, stemming, lemmas, parts of speech and entities with NLTK and spaCy in slides 33-34, sentiment in Part 4, and question answering in Part 6.

In [3]:
import spacy
from nltk.stem import PorterStemmer
from nltk.sentiment import SentimentIntensityAnalyzer

nlp = spacy.load("en_core_web_sm")       # spaCy's small English model
stemmer = PorterStemmer()                # chops word endings by rule
vader = SentimentIntensityAnalyzer()     # positive / negative word list

text = "Jacinda Ardern studied at the University of Waikato. The course was great!"
doc = nlp(text)                          # spaCy runs the whole pipeline here

print("Sentences:")
for sentence in doc.sents:
    print(" ", sentence)

print("\nword, stem, lemma, part of speech:")
for word in doc:
    print(" ", word.text, "|", stemmer.stem(word.text), "|", word.lemma_, "|", word.pos_)

print("\nNamed entities:")
for entity in doc.ents:
    print(" ", entity.text, "->", entity.label_)

print("\nSentiment (-1 negative, +1 positive):")
for sentence in doc.sents:
    print(" ", vader.polarity_scores(sentence.text)["compound"], "->", sentence)

Sentences:
  Jacinda Ardern studied at the University of Waikato.
  The course was great!

word, stem, lemma, part of speech:
  Jacinda | jacinda | Jacinda | PROPN
  Ardern | ardern | Ardern | PROPN
  studied | studi | study | VERB
  at | at | at | ADP
  the | the | the | DET
  University | univers | University | PROPN
  of | of | of | ADP
  Waikato | waikato | Waikato | PROPN
  . | . | . | PUNCT
  The | the | the | DET
  course | cours | course | NOUN
  was | wa | be | AUX
  great | great | great | ADJ
  ! | ! | ! | PUNCT

Named entities:
  Jacinda Ardern -> PERSON
  the University of Waikato -> ORG

Sentiment (-1 negative, +1 positive):
  0.0 -> Jacinda Ardern studied at the University of Waikato.
  0.6588 -> The course was great!


## Slide 7: why it is hard

Each of these headlines has two readings. The words are the same;
what changes is which job one word or phrase is doing.

| Headline | Reading 1 | Reading 2 |
|---|---|---|
| Drunk gets nine years in violin case | nine years in prison, over a court case about a violin | nine years locked inside a violin case |
| Red tape holds up new bridge | paperwork delays the bridge | sticky tape is holding the bridge up |
| Miners refuse to work after death | after a colleague died | after they themselves have died |

A person picks the sensible reading without noticing, using common sense
about prisons and bridges. A computer has no such knowledge unless it
learns it from data. Keep this table in mind: every method in this notebook
is a different attempt at that problem.

## Slide 11: text is stored as numbers

Every character has a number in **Unicode**, written like `U+0041` for `A`.
Unicode 18.0 (September 2026) lists 172,808 characters.

**UTF-8** is the usual way to store those numbers as bytes. It is
*variable width*: a character takes 1, 2, 3 or 4 bytes.

Here is a word with a macron, a space and an emoji. Before I run this: how
many **characters** would you say it has, and how many **bytes** will UTF-8
need?

In [4]:
word = "Māori 👍"

print("characters:", len(word))
print("bytes in UTF-8:", len(word.encode("utf-8")))

characters: 7
bytes in UTF-8: 11


The two numbers differ. One row per character shows where the extra bytes
come from.

In [5]:
rows = []
for character in word:
    rows.append({
        "character": character,
        "code point": "U+" + format(ord(character), "04X"),
        "UTF-8 bytes": len(character.encode("utf-8")),
    })

print(pd.DataFrame(rows))

  character code point  UTF-8 bytes
0         M     U+004D            1
1         ā     U+0101            2
2         o     U+006F            1
3         r     U+0072            1
4         i     U+0069            1
5               U+0020            1
6         👍    U+1F44D            4


Plain English letters take one byte each, the same as the old ASCII
standard. `ā` takes two and the emoji takes four.

**Encoding** turns characters into bytes; **decoding** turns bytes back into
characters. Both sides must use the same table. Before I run this: what do
you think happens if the bytes for "café" are read back with the wrong
table, Latin-1?

In [6]:
stored = "café".encode("utf-8")
print("the bytes:      ", stored)

print("read as UTF-8:  ", stored.decode("utf-8"))
print("read as Latin-1:", stored.decode("latin-1"))

the bytes:       b'caf\xc3\xa9'
read as UTF-8:   café
read as Latin-1: cafÃ©


`cafÃ©` is the classic symptom, and it has a name: **mojibake**. Nothing was
lost; the right bytes were read with the wrong table. `pd.read_csv` reads UTF-8
by default. If `Ã` appears anyway, the text was decoded wrongly somewhere
before it was saved (opening and re-saving a CSV in Excel is a common
cause). Ask the source which encoding it used and pass it, for example
`pd.read_csv(path, encoding="cp1252")`. You will meet this again in Part 2, on a
real website.

Python carries its own copy of the Unicode table. It is older than the
latest standard, which matters only for very new characters.

In [7]:
print("Unicode version known to this Python:", unicodedata.unidata_version)

Unicode version known to this Python: 14.0.0


## Slides 10 and 12: regular expressions, the functions

A **regular expression** (regex) is a short pattern that describes text:
"one or more digits", "a word that starts with a capital letter". Python's
built-in `re` module does the matching.

The **made-up** message below is the text for the next few cells.

In [8]:
# Made up for this notebook. example.com is reserved for examples.
message = ("Order 1043 shipped 03/11/2026. Order 1044 shipped 05/11/2026. "
           "Questions? Email help@example.com or reply here.")
print(message)

Order 1043 shipped 03/11/2026. Order 1044 shipped 05/11/2026. Questions? Email help@example.com or reply here.


`\d` means "a digit" and `+` means "one or more of the thing before". So
`\d+` means "a run of digits".

Before I run this: what will `re.findall(r"\d+", message)` return? Write
the list down.

In [9]:
print(re.findall(r"\d+", message)) #\d = digit, \w = word character, \b = edge of a word.

['1043', '03', '11', '2026', '1044', '05', '11', '2026']


In [10]:
message1 = "Order 42 shipped on day 7, cat catalogue"

print(re.findall(r"\bcat\b", message1))  # ['cat']  correct: whole word "cat" only
print(re.findall("\bcat\b", message1))   # []       wrong: no error, just no matches

['cat']
[]


Each date was cut into three pieces, because `/` is not a digit. A better
pattern says what a date looks like: two digits, a slash, two digits, a
slash, four digits. `\d{4}` means "exactly four digits".

In [11]:
print(re.findall(r"\d\d/\d\d/\d{4}", message))

['03/11/2026', '05/11/2026']


**`search` and `match` are not the same.** `search` looks anywhere in the
text; `match` looks only at the very start. Before I run this: which of the
two finds the word "Email"?

In [12]:
print("search:", re.search(r"Email", message))
print("match: ", re.match(r"Email", message))

search: <re.Match object; span=(73, 78), match='Email'>
match:  None


`match` gave `None` because the message starts with "Order", not "Email".

In [13]:
found = re.search(r"Email", message)

print(found.group())    # the text that matched
print(found.start())    # where it starts
print(found.end())      # where it stops
print(message[73:78])   # same slice, by hand


Email
73
78
Email


**Groups.** Round brackets mark the parts you want back. `group(0)` is the
whole match; `group(1)`, `group(2)` and so on are the bracketed parts.

In [14]:
found = re.search(r"(\d\d)/(\d\d)/(\d{4})", message)

print("whole match:", found.group(0))
print("day:  ", found.group(1))
print("month:", found.group(2))
print("year: ", found.group(3))



whole match: 03/11/2026
day:   03
month: 11
year:  2026


In [15]:
# Get both dates and use groups on each
for found in re.finditer(r"(\d\d)/(\d\d)/(\d{4})", message):
    print("whole match:", found.group(0))
    print("day:  ", found.group(1))
    print("month:", found.group(2))
    print("year: ", found.group(3))

whole match: 03/11/2026
day:   03
month: 11
year:  2026
whole match: 05/11/2026
day:   05
month: 11
year:  2026


**`sub` replaces.** `\S` means "any character that is not a space", so
`\S+@\S+` is "a run of non-spaces, an @, a run of non-spaces": a rough
email address. Hiding personal details before sharing text is a common
real use.

In [16]:
# Made up for this notebook. example.com is reserved for examples.
message = ("Order 1043 shipped 03/11/2026. Order 1044 shipped 05/11/2026. "
           "Questions? Email help@example.com or reply here.")
print(message)

Order 1043 shipped 03/11/2026. Order 1044 shipped 05/11/2026. Questions? Email help@example.com or reply here.


In [17]:
print(re.sub(r"\S+@\S+", "[email hidden]", message))

Order 1043 shipped 03/11/2026. Order 1044 shipped 05/11/2026. Questions? Email [email hidden] or reply here.
